### 1- EQL TRAINING (COMPOSITION DATASET)

In [1]:
# ============================================================
# EQL SYMBOLIC REGRESSION — COMPOSITION
# CONTROLLED OPERATORS
#
# Dataset:
#     composition_dataset_reduced
#
# Inputs:
#     C1
#     C2
#     C3_plus
#     H2O
#     CO2
#     P
#
# Target:
#     HFT
#
# Same grouped frozen split methodology as Paper 1
# ============================================================


# ============================================================
# 1) IMPORTS
# ============================================================

import os
import random
import inspect
import re

import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.model_selection import GroupShuffleSplit

from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score
)

from EQL.model import EQL
from EQL.layer import EqlLayer


# ============================================================
# 2) REPRODUCIBILITY
# ============================================================

SEED = 42

os.environ["PYTHONHASHSEED"] = str(SEED)

np.random.seed(SEED)
random.seed(SEED)
tf.random.set_seed(SEED)


# ============================================================
# 3) FILES / DIRECTORIES
# ============================================================

DATA_FILE = "composition_dataset_reduced.csv"

RESULT_DIR = "results_EQL_COMPOSITION_Controlled"

os.makedirs(
    RESULT_DIR,
    exist_ok=True
)


# ============================================================
# 4) LOAD DATA
# ============================================================

df = pd.read_csv(DATA_FILE)

df = df.rename(columns={
    "Pressure (psia)": "P",
    "Hydrate Temperature (K)": "HFT"
})


# Keep only required columns
feature_cols = [
    "C1",
    "C2",
    "C3_plus",
    "H2O",
    "CO2",
    "P"
]

target_col = "HFT"

df = df[
    feature_cols + [target_col]
]


# Remove invalid values
df = (
    df
    .replace([np.inf, -np.inf], np.nan)
    .dropna()
    .reset_index(drop=True)
)


print("\n============================================================")
print("COMPOSITION DATASET")
print("============================================================")

print(
    f"Total observations : {len(df)}"
)

print(
    "Features           :",
    feature_cols
)

print(
    "Target             :",
    target_col
)

print("\nColumns:")
print(df.columns.tolist())


# ============================================================
# 5) INPUT / TARGET
# ============================================================

X = df[
    feature_cols
].copy()

y = df[
    target_col
].astype(float).values


# ============================================================
# 6) SAME GROUPING PRINCIPLE AS PAPER 1
#
# Round all input variables to 3 decimals
# and construct a combined group identifier.
# ============================================================

round_cols = []

for col in feature_cols:

    round_col = f"{col}_round"

    df[round_col] = df[col].round(3)

    round_cols.append(round_col)


df["group_id"] = (
    df[round_cols]
    .astype(str)
    .agg("_".join, axis=1)
)


groups = df[
    "group_id"
].values


# ============================================================
# 7) FROZEN GROUPED 80/20 SPLIT
# ============================================================

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=SEED
)


train_idx, test_idx = next(
    gss.split(
        X,
        y,
        groups
    )
)


X_train = X.iloc[
    train_idx
].reset_index(drop=True)


X_test = X.iloc[
    test_idx
].reset_index(drop=True)


y_train = y[
    train_idx
]


y_test = y[
    test_idx
]


groups_train = groups[
    train_idx
]


groups_test = groups[
    test_idx
]


# ============================================================
# 8) LEAKAGE CHECK
# ============================================================

overlap = (
    set(groups_train)
    &
    set(groups_test)
)


assert len(overlap) == 0, (
    f"Data leakage detected: "
    f"{len(overlap)} overlapping groups."
)


print("\n============================================================")
print("FROZEN GROUPED SPLIT")
print("============================================================")

print(
    f"Total observations : {len(df)}"
)

print(
    f"Training           : {len(X_train)}"
)

print(
    f"Testing            : {len(X_test)}"
)

print(
    f"Training groups    : "
    f"{len(set(groups_train))}"
)

print(
    f"Testing groups     : "
    f"{len(set(groups_test))}"
)

print(
    f"Group overlap      : {len(overlap)}"
)


# ============================================================
# 9) SAVE FROZEN INDICES
# ============================================================

pd.DataFrame({
    "train_idx": train_idx
}).to_csv(
    os.path.join(
        RESULT_DIR,
        "EQL_COMPOSITION_Controlled_train_idx.csv"
    ),
    index=False
)


pd.DataFrame({
    "test_idx": test_idx
}).to_csv(
    os.path.join(
        RESULT_DIR,
        "EQL_COMPOSITION_Controlled_test_idx.csv"
    ),
    index=False
)


# ============================================================
# 10) METRICS
# ============================================================

def calculate_metrics(
    y_true,
    y_pred
):

    y_true = np.asarray(
        y_true,
        dtype=float
    )

    y_pred = np.asarray(
        y_pred,
        dtype=float
    )


    rmse = np.sqrt(
        mean_squared_error(
            y_true,
            y_pred
        )
    )


    mae = mean_absolute_error(
        y_true,
        y_pred
    )


    r2 = r2_score(
        y_true,
        y_pred
    )


    rrmse = (
        100.0
        * rmse
        / np.mean(np.abs(y_true))
    )


    return {
        "RMSE_K": rmse,
        "MAE_K": mae,
        "RRMSE_%": rrmse,
        "R2": r2
    }


# ============================================================
# 11) STANDARDIZATION
#
# IMPORTANT:
# Mean/std calculated from TRAINING DATA ONLY.
# ============================================================

X_mean = X_train.mean()

X_std = X_train.std()


X_train_scaled = (
    X_train - X_mean
) / X_std


X_test_scaled = (
    X_test - X_mean
) / X_std


X_train_np = (
    X_train_scaled
    .values
    .astype(np.float32)
)


X_test_np = (
    X_test_scaled
    .values
    .astype(np.float32)
)


y_train_np = (
    y_train
    .astype(np.float32)
)


y_test_np = (
    y_test
    .astype(np.float32)
)


print("\n============================================================")
print("STANDARDIZATION")
print("============================================================")

for col in feature_cols:

    print(
        f"{col:10s} "
        f"Mean = {X_mean[col]:.8f}   "
        f"Std = {X_std[col]:.8f}"
    )


# ============================================================
# 12) EQL OPERATOR INSPECTION
# ============================================================

print("\n============================================================")
print("EQL OPERATOR INFORMATION")
print("============================================================")

print(
    inspect.signature(EqlLayer)
)


# ============================================================
# 13) CREATE EQL MODEL
#
# IMPORTANT:
# SP_GR had dim=2
#
# Composition has 6 inputs
# ============================================================

model = EQL(
    num_layers=2,
    dim=6,
    v=[1, 1]
)


# ============================================================
# 14) BUILD / COMPILE
# ============================================================

model.build_and_compile_model()


print("\n============================================================")
print("EQL COMPOSITION MODEL")
print("============================================================")

model.summary()


# ============================================================
# 15) TRAINING
#
# Same training settings as SP_GR EQL
#
# T0 = 2000 epochs
# T1 = 1000 epochs
# T2 = 1000 epochs
#
# lambda = 0.001
# batch_size = 32
# shuffle = True
# ============================================================

print("\n============================================================")
print("EQL COMPOSITION TRAINING")
print("============================================================")


model.fit(
    X_train_np,
    y_train_np,

    lmbda=0.001,

    t0=2000,
    t1=1000,
    t2=1000,

    batch_size=32,

    shuffle=True,

    verbose=0
)


print(
    "EQL Composition training completed."
)


# ============================================================
# 16) PREDICTIONS
# ============================================================

pred_train = np.asarray(
    model.predict(
        X_train_np,
        verbose=0
    )
).ravel()


pred_test = np.asarray(
    model.predict(
        X_test_np,
        verbose=0
    )
).ravel()


# ============================================================
# 17) METRICS
# ============================================================

train_metrics = calculate_metrics(
    y_train,
    pred_train
)


test_metrics = calculate_metrics(
    y_test,
    pred_test
)


print("\n============================================================")
print("EQL COMPOSITION PERFORMANCE")
print("============================================================")


print("\nTRAIN")

for key, value in train_metrics.items():

    print(
        f"{key:12s}: "
        f"{value:.6f}"
    )


print("\nFROZEN TEST")

for key, value in test_metrics.items():

    print(
        f"{key:12s}: "
        f"{value:.6f}"
    )


# ============================================================
# 18) EXTRACT RAW EQL EQUATION
#
# IMPORTANT:
# reduce=False avoids expensive SymPy simplify()
# ============================================================

print("\n============================================================")
print("LEARNED EQL COMPOSITION EQUATION")
print("============================================================")


equation = model.formula(
    raw_latex=False,
    reduce=False
)


print("\nRAW EQL EQUATION:")
print(equation)


# ============================================================
# 19) SAVE RAW EQUATION
# ============================================================

equation_file = os.path.join(
    RESULT_DIR,
    "EQL_COMPOSITION_Controlled_raw_equation.txt"
)


with open(
    equation_file,
    "w",
    encoding="utf-8"
) as f:

    f.write(
        str(equation)
    )


# ============================================================
# 20) SAVE TEST PREDICTIONS
# ============================================================

test_results = X_test.copy()

test_results[
    "HFT_Actual_K"
] = y_test


test_results[
    "HFT_Predicted_K"
] = pred_test


test_results[
    "Error_K"
] = (
    pred_test - y_test
)


test_results[
    "Abs_Error_K"
] = np.abs(
    pred_test - y_test
)


test_results.to_csv(
    os.path.join(
        RESULT_DIR,
        "EQL_COMPOSITION_Controlled_test_predictions.csv"
    ),
    index=False
)


# ============================================================
# 21) SAVE TRAIN PREDICTIONS
# ============================================================

train_results = X_train.copy()

train_results[
    "HFT_Actual_K"
] = y_train


train_results[
    "HFT_Predicted_K"
] = pred_train


train_results[
    "Error_K"
] = (
    pred_train - y_train
)


train_results[
    "Abs_Error_K"
] = np.abs(
    pred_train - y_train
)


train_results.to_csv(
    os.path.join(
        RESULT_DIR,
        "EQL_COMPOSITION_Controlled_train_predictions.csv"
    ),
    index=False
)


# ============================================================
# 22) SAVE SCALING PARAMETERS
# ============================================================

scaling_parameters = pd.DataFrame({

    "Variable": feature_cols,

    "Mean": [
        X_mean[col]
        for col in feature_cols
    ],

    "Std": [
        X_std[col]
        for col in feature_cols
    ]

})


scaling_parameters.to_csv(
    os.path.join(
        RESULT_DIR,
        "EQL_COMPOSITION_Controlled_scaling_parameters.csv"
    ),
    index=False
)


# ============================================================
# 23) SAVE METRICS
# ============================================================

metrics_table = pd.DataFrame([

    {
        "Dataset": "Train",
        **train_metrics
    },

    {
        "Dataset": "Frozen_Test",
        **test_metrics
    }

])


metrics_table.to_csv(
    os.path.join(
        RESULT_DIR,
        "EQL_COMPOSITION_Controlled_metrics.csv"
    ),
    index=False
)


# ============================================================
# 24) FINAL SUMMARY
# ============================================================

print("\n============================================================")
print("FINAL EQL COMPOSITION CONTROLLED SUMMARY")
print("============================================================")


print(
    f"Train RRMSE : "
    f"{train_metrics['RRMSE_%']:.6f} %"
)


print(
    f"Test RRMSE  : "
    f"{test_metrics['RRMSE_%']:.6f} %"
)


print(
    f"Test RMSE   : "
    f"{test_metrics['RMSE_K']:.6f} K"
)


print(
    f"Test MAE    : "
    f"{test_metrics['MAE_K']:.6f} K"
)


print(
    f"Test R²     : "
    f"{test_metrics['R2']:.6f}"
)


print("\nResults saved in:")

print(
    os.path.abspath(
        RESULT_DIR
    )
)


print("\nDone.")


COMPOSITION DATASET
Total observations : 1608
Features           : ['C1', 'C2', 'C3_plus', 'H2O', 'CO2', 'P']
Target             : HFT

Columns:
['C1', 'C2', 'C3_plus', 'H2O', 'CO2', 'P', 'HFT']

FROZEN GROUPED SPLIT
Total observations : 1608
Training           : 1286
Testing            : 322
Training groups    : 1260
Testing groups     : 315
Group overlap      : 0

STANDARDIZATION
C1         Mean = 72.03851645   Std = 20.63813237
C2         Mean = 3.99852992   Std = 4.37211242
C3_plus    Mean = 5.06027378   Std = 5.89469958
H2O        Mean = 2.80707158   Std = 4.29252697
CO2        Mean = 16.05458218   Std = 20.74125524
P          Mean = 1040.83542095   Std = 787.32936451

EQL OPERATOR INFORMATION
(w_initializer, b_initializer, v, lmbda=0, mask=None, exclude=None)

EQL COMPOSITION MODEL
Model: "model"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_1 (InputLayer)        [(None, 6)]             

### EQUATION FUNCTION 

In [2]:
# ============================================================
# EQL RAW EQUATION — COMPOSITION
# DIRECT NUMPY IMPLEMENTATION
#
# x1 = standardized C1
# x2 = standardized C2
# x3 = standardized C3_plus
# x4 = standardized H2O
# x5 = standardized CO2
# x6 = standardized Pressure
#
# No SymPy
# No lambdify
# No simplify
# No parsing
# ============================================================

def EQL_RAW_FUNCTION(
    x1, x2, x3, x4, x5, x6
):

    # ========================================================
    # 1. FIRST LINEAR BRANCH
    # ========================================================

    linear_base = (

        -0.834513676586251 * x1

        -0.135762429372660 * x2

        -0.265554650040848 * x3

        -0.0188043165448891 * x4

        -0.744665985332409 * x5

        +3.50460665820571 * x6
    )


    # ========================================================
    # 2. FIRST MULTIPLICATION TERMS
    # ========================================================

    mult_A = (

        0.111613310873508 * x1

        +0.0168319009244442 * x2

        +0.0350248031318188 * x3

        +0.0960879176855087 * x5

        -1.48341143131256 * x6

        +1.52227199077606
    )


    mult_B = (

        0.0556528083980083 * x1

        +0.0161058045923710 * x2

        +0.0336478687822819 * x3

        -0.00902321934700012 * x4

        +0.0798276066780090 * x5

        +1.68400764465332 * x6

        +1.09841728210449
    )


    mult_AB = mult_A * mult_B


    # ========================================================
    # 3. SIGMOID BASE
    # ========================================================

    sigmoid_base_arg = (

        0.0388400778174400 * x1

        +0.0442386567592621 * x2

        +0.00993176829069853 * x3

        -0.101613037288189 * x4

        -0.112859159708023 * x5

        -1.50128531455994 * x6

        +3.59116625785828
    )


    sigmoid_base = (

        1.0
        /
        (
            1.0
            + np.exp(-sigmoid_base_arg)
        )
    )


    # ========================================================
    # 4. SIN BASE
    # ========================================================

    sin_base_arg = (

        0.141627192497253 * x1

        +0.0582410022616386 * x2

        +0.0740457624197006 * x3

        +0.135016664862633 * x5

        +0.194678097963333 * x6

        -1.79489302635193
    )


    sin_base = np.sin(
        sin_base_arg
    )


    # ========================================================
    # 5. COS BASE
    # ========================================================

    cos_base_arg = (

        0.0280194673687220 * x1

        +0.000637338031083345 * x5

        -1.47866940498352 * x6

        -2.22189831733704
    )


    cos_base = np.cos(
        cos_base_arg
    )


    # ========================================================
    # 6. FIRST NONLINEAR BRANCH
    # ========================================================

    branch_1 = (

        0.319059959136769 * x1

        +0.0519061057754544 * x2

        +0.101529619187559 * x3

        +0.00718946212235877 * x4

        +0.284708454177316 * x5

        -1.33991771319040 * x6

        -0.422356903553009 * mult_AB

        -2.32195448875427 * sigmoid_base

        +2.28084921836853 * sin_base

        -2.28282928466797 * cos_base

        -7.71638752727978
    )


    # ========================================================
    # 7. SECOND NONLINEAR BRANCH
    # ========================================================

    branch_2 = (

        0.331108879718212 * x1

        +0.0538662782391897 * x2

        +0.105363764724231 * x3

        +0.00746096362436471 * x4

        +0.295460130954712 * x5

        -1.39051811493175 * x6

        -0.421435832977295 * mult_AB

        -2.23090744018555 * sigmoid_base

        +2.27155327796936 * sin_base

        -2.25634241104126 * cos_base

        -7.92775526251074
    )


    # ========================================================
    # 8. NESTED SIGMOID
    # ========================================================

    sigmoid_nested_arg = (

        -1.07839943629155 * x1

        -0.175438859077723 * x2

        -0.343162722125925 * x3

        -0.0242998586252176 * x4

        -0.962293850105603 * x5

        +4.52882433286527 * x6

        +4.80826854705811 * mult_AB

        +1.38483560085297 * sigmoid_base

        -1.41269075870514 * sin_base

        -0.134929239749908 * cos_base

        +20.4050056805303
    )


    sigmoid_nested = (

        1.0
        /
        (
            1.0
            + np.exp(-sigmoid_nested_arg)
        )
    )


    # ========================================================
    # 9. NESTED SIN
    # ========================================================

    sin_nested_arg = (

        0.342521167152476 * x1

        +0.0557228803659671 * x2

        +0.108995324135179 * x3

        +0.00771811970393077 * x4

        +0.305643717522038 * x5

        -1.43844492506037 * x6

        -1.90495002269745 * mult_AB

        +0.271582007408142 * sigmoid_base

        -0.491528213024139 * sin_base

        -4.38352918624878 * cos_base

        -5.83760579607122
    )


    sin_nested = np.sin(
        sin_nested_arg
    )


    # ========================================================
    # 10. NESTED COS
    # ========================================================

    cos_nested_arg = (

        0.0843633993028892 * x1

        +0.0137246163374438 * x2

        +0.0268456870231049 * x3

        +0.00190098270382325 * x4

        +0.0752804365350439 * x5

        -0.354290815358752 * x6

        +0.0322199203073978 * mult_AB

        -0.690756738185883 * sigmoid_base

        +0.372169375419617 * sin_base

        -2.20111107826233 * cos_base

        -2.23180638706052
    )


    cos_nested = np.cos(
        cos_nested_arg
    )


    # ========================================================
    # 11. FINAL EQL EQUATION
    # ========================================================

    y = (

        linear_base

        +6.76232234844389
        * mult_AB


        +1.87626731395721
        * branch_1
        * branch_2


        +4.59943196602890
        * sigmoid_base


        +11.0452270507813
        * sigmoid_nested


        -5.16628799326890
        * sin_base


        -8.59634208679199
        * sin_nested


        +2.35973514848736
        * cos_base


        +23.7928791046143
        * cos_nested


        +20.0662635673776
    )

    return y

### EXTERNAL VALIDATION 

In [2]:
# ============================================================
# EQL COMPOSITION — EXTERNAL VALIDATION
# DIRECT NUMPY IMPLEMENTATION
#
# IMPORTANT:
#   - NO retraining
#   - NO new train/test split
#   - NO SymPy
#   - NO lambdify
#   - NO simplify
#
# Composition variables:
#   x1 = standardized C1
#   x2 = standardized C2
#   x3 = standardized C3_plus
#   x4 = standardized H2O
#   x5 = standardized CO2
#   x6 = standardized Pressure
#
# External datasets:
#   1) Levan_3
#   2) Sample_11
#   3) Salman
# ============================================================


import os
import numpy as np
import pandas as pd

from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score
)


# ============================================================
# EQL RAW EQUATION — COMPOSITION
# DIRECT NUMPY IMPLEMENTATION
#
# x1 = standardized C1
# x2 = standardized C2
# x3 = standardized C3_plus
# x4 = standardized H2O
# x5 = standardized CO2
# x6 = standardized Pressure
#
# No SymPy
# No lambdify
# No simplify
# No parsing
# ============================================================

def EQL_RAW_FUNCTION(x1, x2, x3, x4, x5, x6):
    # 1. FIRST LINEAR BRANCH
    linear_base = (
        -0.834513676586251 * x1
        -0.135762429372660 * x2
        -0.265554650040848 * x3
        -0.0188043165448891 * x4
        -0.744665985332409 * x5
        +3.50460665820571 * x6
    )

    # 2. FIRST MULTIPLICATION TERMS
    mult_A = (
        0.111613310873508 * x1
        +0.0168319009244442 * x2
        +0.0350248031318188 * x3
        +0.0960879176855087 * x5
        -1.48341143131256 * x6
        +1.52227199077606
    )
    mult_B = (
        0.0556528083980083 * x1
        +0.0161058045923710 * x2
        +0.0336478687822819 * x3
        -0.00902321934700012 * x4
        +0.0798276066780090 * x5
        +1.68400764465332 * x6
        +1.09841728210449
    )
    mult_AB = mult_A * mult_B

    # 3. SIGMOID BASE
    sigmoid_base_arg = (
        0.0388400778174400 * x1
        +0.0442386567592621 * x2
        +0.00993176829069853 * x3
        -0.101613037288189 * x4
        -0.112859159708023 * x5
        -1.50128531455994 * x6
        +3.59116625785828
    )
    sigmoid_base = 1.0 / (1.0 + np.exp(-sigmoid_base_arg))

    # 4. SIN BASE
    sin_base_arg = (
        0.141627192497253 * x1
        +0.0582410022616386 * x2
        +0.0740457624197006 * x3
        +0.135016664862633 * x5
        +0.194678097963333 * x6
        -1.79489302635193
    )
    sin_base = np.sin(sin_base_arg)

    # 5. COS BASE
    cos_base_arg = (
        0.0280194673687220 * x1
        +0.000637338031083345 * x5
        -1.47866940498352 * x6
        -2.22189831733704
    )
    cos_base = np.cos(cos_base_arg)

    # 6. FIRST NONLINEAR BRANCH
    branch_1 = (
        0.319059959136769 * x1
        +0.0519061057754544 * x2
        +0.101529619187559 * x3
        +0.00718946212235877 * x4
        +0.284708454177316 * x5
        -1.33991771319040 * x6
        -0.422356903553009 * mult_AB
        -2.32195448875427 * sigmoid_base
        +2.28084921836853 * sin_base
        -2.28282928466797 * cos_base
        -7.71638752727978
    )

    # 7. SECOND NONLINEAR BRANCH
    branch_2 = (
        0.331108879718212 * x1
        +0.0538662782391897 * x2
        +0.105363764724231 * x3
        +0.00746096362436471 * x4
        +0.295460130954712 * x5
        -1.39051811493175 * x6
        -0.421435832977295 * mult_AB
        -2.23090744018555 * sigmoid_base
        +2.27155327796936 * sin_base
        -2.25634241104126 * cos_base
        -7.92775526251074
    )

    # 8. NESTED SIGMOID
    sigmoid_nested_arg = (
        -1.07839943629155 * x1
        -0.175438859077723 * x2
        -0.343162722125925 * x3
        -0.0242998586252176 * x4
        -0.962293850105603 * x5
        +4.52882433286527 * x6
        +4.80826854705811 * mult_AB
        +1.38483560085297 * sigmoid_base
        -1.41269075870514 * sin_base
        -0.134929239749908 * cos_base
        +20.4050056805303
    )
    sigmoid_nested = 1.0 / (1.0 + np.exp(-sigmoid_nested_arg))

    # 9. NESTED SIN
    sin_nested_arg = (
        0.342521167152476 * x1
        +0.0557228803659671 * x2
        +0.108995324135179 * x3
        +0.00771811970393077 * x4
        +0.305643717522038 * x5
        -1.43844492506037 * x6
        -1.90495002269745 * mult_AB
        +0.271582007408142 * sigmoid_base
        -0.491528213024139 * sin_base
        -4.38352918624878 * cos_base
        -5.83760579607122
    )
    sin_nested = np.sin(sin_nested_arg)

    # 10. NESTED COS
    cos_nested_arg = (
        0.0843633993028892 * x1
        +0.0137246163374438 * x2
        +0.0268456870231049 * x3
        +0.00190098270382325 * x4
        +0.0752804365350439 * x5
        -0.354290815358752 * x6
        +0.0322199203073978 * mult_AB
        -0.690756738185883 * sigmoid_base
        +0.372169375419617 * sin_base
        -2.20111107826233 * cos_base
        -2.23180638706052
    )
    cos_nested = np.cos(cos_nested_arg)

    # 11. FINAL EQL EQUATION
    y = (
        linear_base
        +6.76232234844389 * mult_AB
        +1.87626731395721 * branch_1 * branch_2
        +4.59943196602890 * sigmoid_base
        +11.0452270507813 * sigmoid_nested
        -5.16628799326890 * sin_base
        -8.59634208679199 * sin_nested
        +2.35973514848736 * cos_base
        +23.7928791046143 * cos_nested
        +20.0662635673776
    )
    return y



# ============================================================
# 1) RESULT DIRECTORY
# ============================================================

if "RESULT_DIR" not in globals():

    RESULT_DIR = "results_EQL_COMPOSITION_Controlled"

    os.makedirs(
        RESULT_DIR,
        exist_ok=True
    )


EXTERNAL_RESULT_DIR = os.path.join(
    RESULT_DIR,
    "external_validation"
)

os.makedirs(
    EXTERNAL_RESULT_DIR,
    exist_ok=True
)


# ============================================================
# 2) COMPOSITION TRAINING SCALING PARAMETERS
#    Explicitly set to the values supplied by the user.
#    These are the ORIGINAL TRAINING-set statistics; do not
#    recalculate them from external validation data.
# ============================================================

required_variables = [
    "C1",
    "C2",
    "C3_plus",
    "H2O",
    "CO2",
    "P"
]

X_mean = pd.Series({
    "C1":      72.03851645,
    "C2":       3.99852992,
    "C3_plus":  5.06027378,
    "H2O":      2.80707158,
    "CO2":     16.05458218,
    "P":     1040.83542095
}, dtype=float)

X_std = pd.Series({
    "C1":      20.63813237,
    "C2":       4.37211242,
    "C3_plus":  5.89469958,
    "H2O":      4.29252697,
    "CO2":     20.74125524,
    "P":      787.32936451
}, dtype=float)

print("\n============================================================")
print("COMPOSITION TRAINING SCALING PARAMETERS")
print("============================================================")
print("Using the fixed training-set means and standard deviations:")

for variable in required_variables:
    print(
        f"{variable:10s} "
        f"Mean = {X_mean[variable]:.10f}   "
        f"Std = {X_std[variable]:.10f}"
    )


# ============================================================
# 3) EXTERNAL DATASETS
# ============================================================

external_datasets = {


    # ========================================================
    # LEVAN_3
    # ========================================================

    "Levan_3": {

        "composition": {

            "CH4": 89.19,
            "CO2": 4.44,
            "C2": 4.24,
            "C3": 1.14,
            "iC4": 0.28,
            "nC4": 0.30,
            "iC5": 0.15,
            "nC5": 0.10,
            "C6": 0.09,
            "C7": 0.07
        },


        "P": np.array([

            690,
            878.33,
            1066.67,
            1255,
            1443.33,
            1631.67,
            1820,
            2008.33,
            2196.67,
            2385,
            2573.33,
            2761.67,
            2950

        ], dtype=float),


        "HFT": np.array([

            285.90,
            287.32,
            288.52,
            289.64,
            290.60,
            291.37,
            291.98,
            292.48,
            292.90,
            293.30,
            293.69,
            294.07,
            294.45

        ], dtype=float)
    },


    # ========================================================
    # SAMPLE 11
    # ========================================================

    "Sample_11": {

        "composition": {

            "CH4": 77.61,
            "CO2": 6.14,
            "C2": 8.81,
            "C3": 3.37,
            "iC4": 0.75,
            "nC4": 1.33,
            "iC5": 0.44,
            "nC5": 0.69,
            "C6": 0.86,
            "C7": 0.00
        },


        "P": np.array([

            590,
            702.22,
            814.44,
            926.67,
            1038.89,
            1151.11,
            1263.33,
            1375.56,
            1487.78,
            1600

        ], dtype=float),


        "HFT": np.array([

            287.94,
            289.30,
            290.32,
            291.09,
            291.66,
            292.12,
            292.54,
            292.98,
            293.53,
            294.26

        ], dtype=float)
    },


    # ========================================================
    # SALMAN
    # ========================================================

    "Salman": {

        "composition": {

            "CH4": 74.10,
            "CO2": 3.95,
            "C2": 10.82,
            "C3": 6.70,
            "iC4": 0.94,
            "nC4": 2.04,
            "iC5": 0.53,
            "nC5": 0.58,
            "C6": 0.25,
            "C7": 0.08
        },


        "P": np.array([

            700,
            891.67,
            1083.33,
            1275,
            1466.67,
            1658.33,
            1850,
            2041.67,
            2233.33,
            2425,
            2616.67,
            2808.33,
            3000

        ], dtype=float),


        "HFT": np.array([

            289.86,
            291.93,
            293.02,
            293.68,
            294.26,
            294.79,
            295.26,
            295.68,
            296.07,
            296.44,
            296.79,
            297.13,
            297.49

        ], dtype=float)
    }
}


# ============================================================
# 4) CONVERT EXTERNAL GAS COMPOSITION
#    TO TRAINING FEATURE SPACE
# ============================================================

def convert_external_composition(composition):


    # --------------------------------------------------------
    # C1 = Methane
    # --------------------------------------------------------

    C1 = composition["CH4"]


    # --------------------------------------------------------
    # C2 = Ethane
    # --------------------------------------------------------

    C2 = composition["C2"]


    # --------------------------------------------------------
    # C3_plus
    #
    # Same definition used for the reduced Composition dataset:
    #
    # C3+
    # = C3 + iC4 + nC4
    #   + iC5 + nC5
    #   + C6 + C7
    # --------------------------------------------------------

    C3_plus = (

        composition["C3"]

        + composition["iC4"]

        + composition["nC4"]

        + composition["iC5"]

        + composition["nC5"]

        + composition["C6"]

        + composition["C7"]
    )


    # --------------------------------------------------------
    # External datasets do not contain water.
    #
    # Therefore:
    #
    # H2O = 0
    # --------------------------------------------------------

    H2O = 0.0


    # --------------------------------------------------------
    # CO2
    # --------------------------------------------------------

    CO2 = composition["CO2"]


    return {

        "C1": C1,
        "C2": C2,
        "C3_plus": C3_plus,
        "H2O": H2O,
        "CO2": CO2
    }


# ============================================================
# 5) METRICS
# ============================================================

def calculate_external_metrics(
    y_true,
    y_pred
):

    y_true = np.asarray(
        y_true,
        dtype=float
    )

    y_pred = np.asarray(
        y_pred,
        dtype=float
    )


    valid = (
        np.isfinite(y_true)
        &
        np.isfinite(y_pred)
    )


    y_true_valid = y_true[valid]

    y_pred_valid = y_pred[valid]


    if len(y_true_valid) == 0:

        raise RuntimeError(
            "No valid prediction points available."
        )


    rmse = np.sqrt(
        mean_squared_error(
            y_true_valid,
            y_pred_valid
        )
    )


    mae = mean_absolute_error(
        y_true_valid,
        y_pred_valid
    )


    # --------------------------------------------------------
    # RRMSE
    #
    # Main metric used in Paper 2
    # --------------------------------------------------------

    rrmse = (

        100.0
        * rmse
        /
        np.mean(
            np.abs(y_true_valid)
        )
    )


    r2 = r2_score(
        y_true_valid,
        y_pred_valid
    )


    error = (
        y_pred_valid
        - y_true_valid
    )


    mean_error = np.mean(
        error
    )


    max_abs_error = np.max(
        np.abs(error)
    )


    return {

        "RMSE_K": rmse,

        "MAE_K": mae,

        "RRMSE_%": rrmse,

        "R2": r2,

        "Mean_Error_K": mean_error,

        "Max_Abs_Error_K": max_abs_error,

        "N_valid": len(
            y_true_valid
        ),

        "N_invalid": (
            len(y_true)
            -
            len(y_true_valid)
        )
    }


# ============================================================
# 6) EXTERNAL VALIDATION
# ============================================================

all_metrics = []

all_predictions = []

dataset_summary = []


print("\n============================================================")
print("EQL COMPOSITION EXTERNAL VALIDATION")
print("============================================================")


for dataset_name, dataset in external_datasets.items():


    print("\n------------------------------------------------------------")
    print(dataset_name)
    print("------------------------------------------------------------")


    # --------------------------------------------------------
    # Pressure
    # --------------------------------------------------------

    P_values = np.asarray(
        dataset["P"],
        dtype=float
    )


    # --------------------------------------------------------
    # Actual HFT
    # --------------------------------------------------------

    y_true = np.asarray(
        dataset["HFT"],
        dtype=float
    )


    # --------------------------------------------------------
    # Dimension check
    # --------------------------------------------------------

    if len(P_values) != len(y_true):

        raise ValueError(
            f"{dataset_name}: "
            f"Pressure and HFT lengths do not match."
        )


    # ========================================================
    # CONVERT COMPOSITION
    # ========================================================

    composition_features = convert_external_composition(
        dataset["composition"]
    )


    C1_value = composition_features["C1"]

    C2_value = composition_features["C2"]

    C3_plus_value = composition_features["C3_plus"]

    H2O_value = composition_features["H2O"]

    CO2_value = composition_features["CO2"]


    print(
        f"C1       : {C1_value:.6f} %"
    )

    print(
        f"C2       : {C2_value:.6f} %"
    )

    print(
        f"C3_plus  : {C3_plus_value:.6f} %"
    )

    print(
        f"H2O      : {H2O_value:.6f} %"
    )

    print(
        f"CO2      : {CO2_value:.6f} %"
    )


    # ========================================================
    # TRAINING-ONLY STANDARDIZATION
    # ========================================================

    C1_scaled = (

        C1_value
        -
        X_mean["C1"]

    ) / X_std["C1"]


    C2_scaled = (

        C2_value
        -
        X_mean["C2"]

    ) / X_std["C2"]


    C3_plus_scaled = (

        C3_plus_value
        -
        X_mean["C3_plus"]

    ) / X_std["C3_plus"]


    H2O_scaled = (

        H2O_value
        -
        X_mean["H2O"]

    ) / X_std["H2O"]


    CO2_scaled = (

        CO2_value
        -
        X_mean["CO2"]

    ) / X_std["CO2"]


    P_scaled = (

        P_values
        -
        X_mean["P"]

    ) / X_std["P"]


    # --------------------------------------------------------
    # Create vectors
    # --------------------------------------------------------

    C1_array = np.full(
        len(P_values),
        C1_scaled,
        dtype=float
    )


    C2_array = np.full(
        len(P_values),
        C2_scaled,
        dtype=float
    )


    C3_plus_array = np.full(
        len(P_values),
        C3_plus_scaled,
        dtype=float
    )


    H2O_array = np.full(
        len(P_values),
        H2O_scaled,
        dtype=float
    )


    CO2_array = np.full(
        len(P_values),
        CO2_scaled,
        dtype=float
    )


    # ========================================================
    # EQL PREDICTION
    #
    # x1 = C1
    # x2 = C2
    # x3 = C3_plus
    # x4 = H2O
    # x5 = CO2
    # x6 = Pressure
    # ========================================================

    try:

        y_pred = EQL_RAW_FUNCTION(

            C1_array,

            C2_array,

            C3_plus_array,

            H2O_array,

            CO2_array,

            P_scaled
        )


    except Exception as error:

        raise RuntimeError(

            f"Prediction failed for "
            f"{dataset_name}: {error}"
        )


    y_pred = np.asarray(
        y_pred,
        dtype=float
    ).reshape(-1)


    # --------------------------------------------------------
    # Prediction length check
    # --------------------------------------------------------

    if len(y_pred) != len(y_true):

        raise ValueError(

            f"{dataset_name}: "
            f"Prediction length does not match "
            f"HFT length."
        )


    # ========================================================
    # METRICS: ALL PRESSURES vs PRESSURE < 2500 psia
    # Predictions are calculated once for every point. Metrics
    # are then evaluated on (1) the complete dataset and
    # (2) only points with pressure strictly below 2500 psia.
    # ========================================================

    pressure_masks = {
        "All_Data": np.ones(len(P_values), dtype=bool),
        "P_below_2500_psia": P_values < 2500.0
    }

    dataset_metric_results = {}

    for pressure_scope, pressure_mask in pressure_masks.items():
        if not np.any(pressure_mask):
            print(f"WARNING: {dataset_name} has no points for {pressure_scope}.")
            continue

        scope_metrics = calculate_external_metrics(
            y_true[pressure_mask],
            y_pred[pressure_mask]
        )
        scope_metrics["Sample"] = dataset_name
        scope_metrics["Pressure_Scope"] = pressure_scope
        scope_metrics["Pressure_Cutoff_psia"] = (
            "All pressures" if pressure_scope == "All_Data"
            else "P < 2500 psia"
        )
        scope_metrics["Pressure_Min_psia"] = float(np.min(P_values[pressure_mask]))
        scope_metrics["Pressure_Max_psia"] = float(np.max(P_values[pressure_mask]))
        dataset_metric_results[pressure_scope] = scope_metrics
        all_metrics.append(scope_metrics)

    # Use full-dataset metrics in the existing console summary.
    metrics = dataset_metric_results.get("All_Data", {})


    # ========================================================
    # ERRORS
    # ========================================================

    error = (

        y_pred
        -
        y_true
    )


    abs_error = np.abs(
        error
    )


    # ========================================================
    # POINT-BY-POINT PREDICTIONS
    # ========================================================

    prediction_table = pd.DataFrame({

        "Sample":
            dataset_name,

        "C1":
            C1_value,

        "C2":
            C2_value,

        "C3_plus":
            C3_plus_value,

        "H2O":
            H2O_value,

        "CO2":
            CO2_value,

        "Pressure_psia":
            P_values,

        "Included_in_P_below_2500":
            P_values < 2500.0,

        "HFT_Actual_K":
            y_true,

        "HFT_Predicted_K":
            y_pred,

        "Error_K":
            error,

        "Abs_Error_K":
            abs_error
    })


    all_predictions.append(
        prediction_table
    )


    # ========================================================
    # DATASET SUMMARY
    # ========================================================

    dataset_summary.append({

        "Sample":
            dataset_name,

        "C1":
            C1_value,

        "C2":
            C2_value,

        "C3_plus":
            C3_plus_value,

        "H2O":
            H2O_value,

        "CO2":
            CO2_value,

        "N_points":
            len(P_values),

        "Pressure_Min_psia":
            np.min(P_values),

        "Pressure_Max_psia":
            np.max(P_values),

        "HFT_Min_K":
            np.min(y_true),

        "HFT_Max_K":
            np.max(y_true)
    })


    # ========================================================
    # PRINT RESULTS
    # ========================================================

    print(
        f"RMSE        : "
        f"{metrics['RMSE_K']:.6f} K"
    )


    print(
        f"RRMSE       : "
        f"{metrics['RRMSE_%']:.6f} %"
    )


    print(
        f"MAE         : "
        f"{metrics['MAE_K']:.6f} K"
    )


    print(
        f"R²          : "
        f"{metrics['R2']:.6f}"
    )


    print(
        f"Mean Error  : "
        f"{metrics['Mean_Error_K']:.6f} K"
    )


    print(
        f"Max Abs Err : "
        f"{metrics['Max_Abs_Error_K']:.6f} K"
    )


# ============================================================
# 7) COMBINE RESULTS
# ============================================================

metrics_df = pd.DataFrame(
    all_metrics
)


predictions_df = pd.concat(
    all_predictions,
    ignore_index=True
)


summary_df = pd.DataFrame(
    dataset_summary
)


# ============================================================
# 8) SAVE METRICS AND SIDE-BY-SIDE COMPARISON
# ============================================================

# Long-format metrics table: one row per dataset and pressure scope.
metrics_file = os.path.join(
    EXTERNAL_RESULT_DIR,
    "EQL_Composition_External_Metrics.csv"
)
metrics_df.to_csv(metrics_file, index=False)

# Wide comparison table: metrics for all pressures beside metrics
# for P < 2500 psia, making the effect of the pressure cutoff easy
# to compare within each external dataset.
comparison_metric_columns = [
    "RMSE_K", "MAE_K", "RRMSE_%", "R2",
    "Mean_Error_K", "Max_Abs_Error_K", "N_valid", "N_invalid"
]
comparison_rows = []
for sample_name in external_datasets:
    sample_rows = metrics_df[metrics_df["Sample"] == sample_name]
    row = {"Sample": sample_name}
    for scope, suffix in [
        ("All_Data", "All_Data"),
        ("P_below_2500_psia", "P_below_2500_psia")
    ]:
        scope_row = sample_rows[sample_rows["Pressure_Scope"] == scope]
        for metric_name in comparison_metric_columns:
            output_name = f"{metric_name}_{suffix}"
            row[output_name] = (
                scope_row.iloc[0][metric_name]
                if not scope_row.empty else np.nan
            )
    # Difference is filtered-subset minus full-dataset metric.
    for metric_name in ["RMSE_K", "MAE_K", "RRMSE_%", "R2"]:
        row[f"Delta_{metric_name}_Filtered_minus_All"] = (
            row.get(f"{metric_name}_P_below_2500_psia", np.nan)
            - row.get(f"{metric_name}_All_Data", np.nan)
        )
    comparison_rows.append(row)

metrics_comparison_df = pd.DataFrame(comparison_rows)
comparison_file = os.path.join(
    EXTERNAL_RESULT_DIR,
    "EQL_Composition_External_Metrics_Comparison.csv"
)
metrics_comparison_df.to_csv(comparison_file, index=False)

print("\n============================================================")
print("METRICS COMPARISON: ALL DATA vs P < 2500 psia")
print("============================================================")
print(metrics_df[[
    "Sample", "Pressure_Scope", "N_valid", "RMSE_K", "MAE_K",
    "RRMSE_%", "R2", "Mean_Error_K", "Max_Abs_Error_K"
]].to_string(index=False, float_format=lambda value: f"{value:.6f}"))
print(f"\nMetrics table saved to: {metrics_file}")
print(f"Side-by-side comparison saved to: {comparison_file}")


# ============================================================
# 9) SAVE POINT-BY-POINT PREDICTIONS
# ============================================================

predictions_file = os.path.join(

    EXTERNAL_RESULT_DIR,

    "EQL_Composition_External_Predictions.csv"
)


predictions_df.to_csv(
    predictions_file,
    index=False
)


# ============================================================
# 10) SAVE DATASET SUMMARY
# ============================================================

summary_file = os.path.join(

    EXTERNAL_RESULT_DIR,

    "EQL_Composition_External_Dataset_Summary.csv"
)


summary_df.to_csv(
    summary_file,
    index=False
)


# ============================================================
# 11) SAVE SCALING PARAMETERS
# ============================================================

scaling_file = os.path.join(

    EXTERNAL_RESULT_DIR,

    "EQL_Composition_External_Scaling_Parameters.csv"
)


external_scaling = pd.DataFrame({

    "Variable":
        required_variables,

    "Training_Mean":
        [
            X_mean[v]
            for v in required_variables
        ],

    "Training_Std":
        [
            X_std[v]
            for v in required_variables
        ]
})


external_scaling.to_csv(
    scaling_file,
    index=False
)


# ============================================================
# 12) FINAL RESULTS TABLE
# ============================================================

print("\n============================================================")
print("FINAL EQL COMPOSITION EXTERNAL VALIDATION RESULTS")
print("============================================================")


print(

    metrics_df[

        [
            "Sample",
            "Pressure_Scope",
            "Pressure_Min_psia",
            "Pressure_Max_psia",
            "RMSE_K",
            "RRMSE_%",
            "MAE_K",
            "R2",
            "Mean_Error_K",
            "Max_Abs_Error_K",
            "N_valid",
            "N_invalid"
        ]

    ].to_string(

        index=False,

        float_format=lambda x:
            f"{x:.6f}"
    )
)


# ============================================================
# 13) POINT-BY-POINT EXTERNAL PREDICTIONS
# ============================================================

print("\n============================================================")
print("POINT-BY-POINT EXTERNAL PREDICTIONS")
print("============================================================")


print(

    predictions_df[

        [
            "Sample",
            "Pressure_psia",
            "HFT_Actual_K",
            "HFT_Predicted_K",
            "Error_K",
            "Abs_Error_K"
        ]

    ].to_string(

        index=False,

        float_format=lambda x:
            f"{x:.4f}"
    )
)


# ============================================================
# 14) FILES
# ============================================================

print("\n============================================================")
print("FILES SAVED")
print("============================================================")


print(
    os.path.abspath(
        EXTERNAL_RESULT_DIR
    )
)


print(
    "\n1. EQL_Composition_External_Metrics.csv"
)

print(
    "2. EQL_Composition_External_Predictions.csv"
)

print(
    "3. EQL_Composition_External_Dataset_Summary.csv"
)

print(
    "4. EQL_Composition_External_Scaling_Parameters.csv"
)
print(
    "5. EQL_Composition_External_Metrics_Comparison.csv"
)


print("\n============================================================")
print("EQL COMPOSITION EXTERNAL VALIDATION COMPLETED")
print("============================================================")


COMPOSITION TRAINING SCALING PARAMETERS
Using the fixed training-set means and standard deviations:
C1         Mean = 72.0385164500   Std = 20.6381323700
C2         Mean = 3.9985299200   Std = 4.3721124200
C3_plus    Mean = 5.0602737800   Std = 5.8946995800
H2O        Mean = 2.8070715800   Std = 4.2925269700
CO2        Mean = 16.0545821800   Std = 20.7412552400
P          Mean = 1040.8354209500   Std = 787.3293645100

EQL COMPOSITION EXTERNAL VALIDATION

------------------------------------------------------------
Levan_3
------------------------------------------------------------
C1       : 89.190000 %
C2       : 4.240000 %
C3_plus  : 2.130000 %
H2O      : 0.000000 %
CO2      : 4.440000 %
RMSE        : 12.199729 K
RRMSE       : 4.188781 %
MAE         : 4.416028 K
R²          : -21.070510
Mean Error  : -2.316416 K
Max Abs Err : 43.675274 K

------------------------------------------------------------
Sample_11
------------------------------------------------------------
C1       : 77